# 改错题 · 时序预测陷阱

时序方向的错误有个统一的名字：**未来信息泄漏**（look-ahead bias）。

它比普通数据泄漏更阴 —— 因为**很多写法看起来天经地义**，
效果还特别好（因为在偷看答案）。等到真实预测时，一切都崩了。


## 一句话判据

**「站在预测时刻 t，这个数字我现在拿得到吗？」**

- 移动平均的窗口里有没有 t 当天？
- 缺失值是用**之前**的值填的，还是之后的？
- 多步预测时，历史序列的最后一个点是不是原点本身？
- 目标时刻的日历特征，有没有和原点的列重名？

> 改完对 `_solution.ipynb`。这套坑与 `05_timeseries/final` 的实测结论一致。


### 题 1 · rolling 忘了 shift

想造一个「用过去 3 天算今天」的移动平均特征。

跑一遍看看：结果 `[nan, nan, 2.0, 3.0, 4.0]` —— 第 3 个值（2.0）已经含当天了。

In [ ]:

import pandas as pd

s = pd.Series([1.0, 2.0, 3.0, 4.0, 5.0], name="负荷")

s_ma = s.rolling(3).mean()

print(s_ma.tolist())


In [ ]:

# 修复后请在这里重写（可以直接改上面那个 cell，也可以写在这里）


### 题 2 · 递归预测的历史漏了原点

要从原点 t=5 往后滚动预测。

跑一遍看看：历史长度是 `5`，最后一个值是 `4.0` —— **原点 5.0 丢了**。

In [ ]:

yv = [float(i) for i in range(10)]
origin = 5

history = yv[:origin]

print("历史序列 =", history)
print("长度 =", len(history))


In [ ]:

# 修复后请在这里重写（可以直接改上面那个 cell，也可以写在这里）


### 题 3 · 直接法的特征列名撞车

把原点特征与目标时刻日历特征横向拼接。

跑一遍看看：列名变成 `['h_sin', 'lag1', 'h_sin']`，两个同名。

In [ ]:

import pandas as pd

origin_feat = pd.DataFrame({"h_sin": [0.1, 0.2], "lag1": [1.0, 2.0]})
target_cal = pd.DataFrame({"h_sin": [0.9, 0.8]})

X = pd.concat([origin_feat, target_cal], axis=1)

print(X.columns.tolist())


In [ ]:

# 修复后请在这里重写（可以直接改上面那个 cell，也可以写在这里）


### 题 4 · 缺失值用 bfill（借了未来）

用 `bfill` 填缺失值。

跑一遍看看：第 2、3 位都被填成 `4.0` —— 那是未来的值。

In [ ]:

import numpy as np
import pandas as pd

s = pd.Series([1.0, np.nan, np.nan, 4.0])

s_filled = s.bfill()

print(s_filled.tolist())


In [ ]:

# 修复后请在这里重写（可以直接改上面那个 cell，也可以写在这里）
